<a href="https://colab.research.google.com/github/[ORGANIZATION]/[REPOSITORY]/blob/main/[PATH]/[NOTEBOOK].ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>

# Accelerating innovation with curated data products: From data assets to data products

## 1. Problem statement and executive summary
- **Target audience**: Data platform engineers, governance architects, and analytics producers building on Google Cloud Knowledge Catalog.
- **Core challenge and solution**: Traditionally, enterprise data assets exist as fragmented silos with disconnected access controls and implicit schemas. In this workflow, producing teams package certified customer churn assets into a single discoverable logical unit—a data product (`type=DATA_PRODUCT`)—governed by machine-readable data contract SLAs (`contract_sla.yaml`) and enriched with golden queries. This enables both human analysts and AI agents (**Gemini Enterprise Agent Platform (`gemini-3.6-flash`)**) to discover, evaluate, and query certified assets without data duplication or SQL hallucination.

## 2. Measurable learning objectives
1. Programmatically instantiate a logical data product entry in Knowledge Catalog bundling multi-modal tables and models without copying data.
2. Author, attach, and validate machine-readable data contract SLAs (refresh cadence and freshness thresholds) as custom catalog aspect types.
3. Discover certified data products via the search API and ground an AI agent on the packaged context to generate verified analytical queries.

## 3. Technical stack and sample data assets
- **AI model**: Gemini Enterprise Agent Platform (`gemini-3.6-flash`)
- **Sample data asset**: gs://sample-churn-telemetry (illustrative sample dataset)
- **Note**: Sample data used purely for educational illustration.

## 4. Architecture outline
- **Section 1**: Introductory overview, target persona, and measurable learning objectives
- **Section 2**: Environment setup and fail-fast parameter validation
- **Section 3**: Reusable helper functions and catalog management architecture
- **Section 4**: Step-by-step educational execution
- **Section 5**: Round-trip assertions and resource cleanup

## 2. Environment setup and parameterized configuration

In the following cell, we install the required SDKs (`google-cloud-dataplex`, `google-genai`, `tabulate`, `pyyaml`) without modifying Colab pre-installed packages such as `pandas` or `google-auth`. We then configure fail-fast interactive parameters to ensure valid project metadata before execution.

In [ ]:
import sys

# Install required SDKs while protecting pre-installed Colab dependencies
!{sys.executable} -m pip install -q google-cloud-dataplex google-genai tabulate pyyaml


### 2.1 Interactive parameter configuration and fail-fast validation

Configure your project ID, region, and catalog identifiers in the following cell. The script enforces fail-fast validation and raises an explicit `ValueError` immediately if any placeholder string is unmodified.

In [ ]:
PROJECT_ID = "your-gcp-project-id"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}
ENTRY_GROUP_ID = "churn_data_products_group"  # @param {type:"string"}
DATA_PRODUCT_ID = "customer_churn_data_product"  # @param {type:"string"}

if not PROJECT_ID or PROJECT_ID == "your-gcp-project-id":
  raise ValueError(
      "Missing required PROJECT_ID: Please enter a valid Google Cloud Project"
      " ID in the @param form before executing."
  )
if not ENTRY_GROUP_ID or ENTRY_GROUP_ID == "your-entry-group-id":
  raise ValueError(
      "Missing required ENTRY_GROUP_ID: Please enter a valid entry group"
      " identifier."
  )
if not DATA_PRODUCT_ID or DATA_PRODUCT_ID == "your-data-product-id":
  raise ValueError(
      "Missing required DATA_PRODUCT_ID: Please enter a valid data product"
      " identifier."
  )

print(f"Verified environment parameters for project: {PROJECT_ID} ({LOCATION})")
print(
    f"Target entry group: {ENTRY_GROUP_ID} | Data product ID: {DATA_PRODUCT_ID}"
)


## 3. Reusable helper functions and architecture

To maintain modularity and keep cells under 80 lines, we encapsulate catalog operations into three dedicated service classes:
1. `DataProductManager`: Handles the creation of catalog entry groups, custom aspect types (`contract_sla` and `golden_queries`), and logical data product entries that bundle multi-modal assets without copying data.
2. `ContractSlaValidator`: Validates asset freshness against SLA thresholds defined in the contract aspect.
3. `AgentGroundingService`: Performs search API discovery and grounds AI agents using structured schemas.

Note on data sovereignty and region selection: Using `location="global"` offers development quota flexibility, but production workloads requiring strict data sovereignty and compliance mandate regional endpoints matching Dataplex (e.g., `us-central1`).

### 3.1 Catalog manager class

The `DataProductManager` class encapsulates catalog API interactions for managing entry groups, aspect types, and logical data products.

In [ ]:
from google.api_core import exceptions as gcp_exceptions
from google.cloud import dataplex_v1


class DataProductManager:
  """Manages catalog entry groups, aspect types, entry types, and data products."""

  def __init__(self, project_id: str, location: str):
    self.project_id = project_id
    self.location = location
    self.client = dataplex_v1.CatalogServiceClient()
    self.parent = f"projects/{project_id}/locations/{location}"

  def setup_entry_group(self, entry_group_id: str, description: str) -> str:
    """Creates or retrieves a catalog entry group."""
    name = f"{self.parent}/entryGroups/{entry_group_id}"
    try:
      group = dataplex_v1.EntryGroup(
          name=name, description=description, display_name="Churn Data Products"
      )
      self.client.create_entry_group(
          parent=self.parent,
          entry_group_id=entry_group_id,
          entry_group=group,
      )
      print(f"Resource setup: Created entry group [{entry_group_id}]")
    except gcp_exceptions.AlreadyExists:
      print(f"Resource setup: Found existing entry group [{entry_group_id}]")
    return name

  def setup_aspect_type(
      self, aspect_type_id: str, display_name: str, metadata_template: dict
  ) -> str:
    """Creates or retrieves a custom aspect type."""
    name = f"{self.parent}/aspectTypes/{aspect_type_id}"
    try:
      aspect = dataplex_v1.AspectType(
          name=name,
          display_name=display_name,
          description=f"Aspect type for {display_name}",
          metadata_template=metadata_template,
      )
      self.client.create_aspect_type(
          parent=self.parent,
          aspect_type_id=aspect_type_id,
          aspect_type=aspect,
      )
      print(f"Resource setup: Created aspect type [{aspect_type_id}]")
    except gcp_exceptions.AlreadyExists:
      print(f"Resource setup: Found existing aspect type [{aspect_type_id}]")
    return name

  def setup_entry_type(
      self, entry_type_id: str, display_name: str, description: str
  ) -> str:
    """Creates or retrieves a custom entry type for data products."""
    name = f"{self.parent}/entryTypes/{entry_type_id}"
    try:
      entry_type = dataplex_v1.EntryType(
          name=name,
          display_name=display_name,
          description=description,
      )
      self.client.create_entry_type(
          parent=self.parent,
          entry_type_id=entry_type_id,
          entry_type=entry_type,
      )
      print(f"Resource setup: Created entry type [{entry_type_id}]")
    except gcp_exceptions.AlreadyExists:
      print(f"Resource setup: Found existing entry type [{entry_type_id}]")
    return name

  def delete_resource_quietly(self, resource_name: str, is_entry: bool = False):
    """Deletes a catalog resource during cleanup."""
    try:
      if is_entry:
        self.client.delete_entry(name=resource_name)
      elif "aspectTypes/" in resource_name:
        self.client.delete_aspect_type(name=resource_name)
      elif "entryTypes/" in resource_name:
        self.client.delete_entry_type(name=resource_name)
      print(f"Resource cleanup: Deleted [{resource_name}]")
    except gcp_exceptions.NotFound:
      pass


### 3.2 Machine-readable data contract SLA validator

The `ContractSlaValidator` class inspects the attached contract SLA aspect of a data product and verifies whether the underlying assets meet the agreed-upon freshness threshold (`max_freshness_hours`) and schema stability guarantees before analytical consumption.

In [ ]:
import datetime
from typing import Dict, Tuple


class ContractSlaValidator:
  """Validates whether a data product asset complies with its SLA terms."""

  @staticmethod
  def validate_freshness_sla(
      contract_aspect: Dict, last_updated_utc: datetime.datetime
  ) -> Tuple[bool, str, float]:
    """Checks asset freshness against the SLA aspect threshold."""
    max_hours = contract_aspect.get("max_freshness_hours", 24.0)
    now_utc = datetime.datetime.now(datetime.timezone.utc)

    if last_updated_utc.tzinfo is None:
      last_updated_utc = last_updated_utc.replace(tzinfo=datetime.timezone.utc)

    age_hours = (now_utc - last_updated_utc).total_seconds() / 3600.0
    is_compliant = age_hours <= max_hours

    status_msg = (
        f"PASSED: Asset freshness ({age_hours:.1f}h) is within SLA"
        f" ({max_hours:.1f}h)"
        if is_compliant
        else (
            f"VIOLATION: Asset freshness ({age_hours:.1f}h) exceeded SLA"
            f" threshold ({max_hours:.1f}h)"
        )
    )
    return is_compliant, status_msg, age_hours

  @staticmethod
  def format_sla_report(
      product_id: str, is_compliant: bool, status_msg: str, schema_stable: bool
  ) -> str:
    """Formats a concise SLA audit report."""
    state = "COMPLIANT" if (is_compliant and schema_stable) else "BREACHED"
    report_lines = [
        f"=== SLA audit report: [{product_id}] ===",
        f"Overall status  : {state}",
        f"Freshness check : {status_msg}",
        f"Schema stability: {'GUARANTEED' if schema_stable else 'UNSTABLE'}",
        "============================================",
    ]
    return "\n".join(report_lines)


### 3.3 AI agent grounding service with structured output

The `AgentGroundingService` class formats the packaged data product context (contract SLA terms, golden queries, and glossary definitions) and invokes the generative model using a Pydantic schema to ensure type-safe SQL generation without schema hallucination.

Note on model availability and region placement: The generative model is invoked in the configured location (`us-central1`). When deploying in production, ensure the selected region supports structured schema extraction, or route requests to supported regional endpoints to comply with data residency boundaries.

In [ ]:
import json
from google import genai
from google.genai import types
from pydantic import BaseModel, Field


class GroundedSqlResponse(BaseModel):
  """Structured output schema for agent-grounded SQL generation."""

  sql_query: str = Field(
      description="The generated BigQuery SQL query adhering to contract SLA."
  )
  explanation: str = Field(
      description=(
          "Why this query conforms to authoritative glossary terms and golden "
          "queries."
      )
  )
  confidence_score: float = Field(
      description="Confidence score between 0.0 and 1.0."
  )


class AgentGroundingService:
  """Grounds AI agents on packaged data product context for SQL generation."""

  def __init__(self, project_id: str, location: str):
    self.project_id = project_id
    self.location = location
    self.client = genai.Client(vertexai=True, project=project_id, location="global")

  def generate_grounded_sql(
      self, product_context: dict, user_question: str
  ) -> GroundedSqlResponse:
    """Generates structured BigQuery SQL using the product's packaged context."""
    prompt_lines = [
        "You are an expert SQL analyst grounded on a certified Google Cloud data product.",
        "DATA PRODUCT CONTEXT:",
        json.dumps(product_context, indent=2),
        "",
        f"USER QUESTION: {user_question}",
        "",
        "Generate a valid BigQuery SQL query using only the bundled tables, glossary definitions, and golden queries provided.",
    ]
    prompt = "\n".join(prompt_lines)

    response = self.client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=GroundedSqlResponse,
            temperature=0.1,
        ),
    )
    return GroundedSqlResponse.model_validate_json(response.text)


## 4. Step-by-step educational execution

In this section, we walk through the end-to-end producer and consumer journeys for Knowledge Catalog Use Case 5.

### 4.1 Authoring machine-readable aspect types for data contract SLAs and golden queries

The producing team first defines custom aspect type templates in Knowledge Catalog to codify machine-readable data contract SLAs (`contract_sla.yaml`) and golden queries. These templates ensure that freshness thresholds and reference SQL patterns travel with the data product.

In [ ]:
# Initialize the catalog manager
manager = DataProductManager(project_id=PROJECT_ID, location=LOCATION)

# Setup custom entry type for logical data products
data_product_entry_type = manager.setup_entry_type(
    entry_type_id="data-product-type",
    display_name="Data Product",
    description="Custom entry type for logical data products",
)

# Define machine-readable data contract SLA template using valid protobuf keys and index
contract_sla_template = {
    "type_": "RECORD",
    "name": "ContractSla",
    "record_fields": [
        {
            "name": "refresh_cadence_cron",
            "type_": "STRING",
            "index": 1,
        },
        {
            "name": "expected_delivery_utc",
            "type_": "STRING",
            "index": 2,
        },
        {
            "name": "max_freshness_hours",
            "type_": "DOUBLE",
            "index": 3,
        },
        {
            "name": "schema_stability_guarantee",
            "type_": "BOOL",
            "index": 4,
        },
    ],
}

# Define golden queries template using valid protobuf keys and index
golden_queries_template = {
    "type_": "RECORD",
    "name": "GoldenQueries",
    "record_fields": [
        {
            "name": "query_title",
            "type_": "STRING",
            "index": 1,
        },
        {
            "name": "sql_template",
            "type_": "STRING",
            "index": 2,
        },
        {
            "name": "business_glossary_term",
            "type_": "STRING",
            "index": 3,
        },
    ],
}

# Setup aspect types in Knowledge Catalog
contract_aspect_name = manager.setup_aspect_type(
    aspect_type_id="contract-sla-aspect",
    display_name="Data Contract SLA",
    metadata_template=contract_sla_template,
)
golden_aspect_name = manager.setup_aspect_type(
    aspect_type_id="golden-queries-aspect",
    display_name="AI Golden Queries",
    metadata_template=golden_queries_template,
)


### 4.2 Packaging the customer churn data product without data copying

The producing team bundles related assets—including the certified BigQuery churn feature table, views, a BigQuery ML churn prediction model, and a Cloud Storage sample telemetry bucket—into a single logical data product entry (`customer_churn_data_product`). This creates a discoverable unit of verified truth without copying physical data. We also declare accountable ownership directly on the entry.

In [ ]:
# Setup parent entry group for churn data products
group_name = manager.setup_entry_group(
    entry_group_id=ENTRY_GROUP_ID,
    description="Governed logical data products for customer churn analytics",
)

# Define logical data product entry bundling multi-modal assets
product_entry_name = f"{group_name}/entries/{DATA_PRODUCT_ID}"
bundled_assets = [
    "bigquery_table:projects/bigquery-public-data/datasets/ml_datasets/tables/credit_card_default",
    f"bigquery_table:projects/{PROJECT_ID}/datasets/churn_prod/tables/customer_features",
    f"bigquery_model:projects/{PROJECT_ID}/datasets/churn_prod/models/churn_prediction_bqml",
]

try:
  entry = dataplex_v1.Entry(
      name=product_entry_name,
      entry_type=data_product_entry_type,
      fully_qualified_name=(
          f"custom:projects/{PROJECT_ID}/dataProducts/{DATA_PRODUCT_ID}"
      ),
      aspects={},
  )
  manager.client.create_entry(
      parent=group_name, entry_id=DATA_PRODUCT_ID, entry=entry
  )
  print(f"Resource setup: Packaged data product [{DATA_PRODUCT_ID}].")
except gcp_exceptions.AlreadyExists:
  print(f"Resource setup: Found existing data product [{DATA_PRODUCT_ID}].")

print("Accountable owner: data-governance-lead@example.com")
print("Designated access approver group: churn-product-approvers@example.com")
print("Bundled assets (no data copying):")
for asset in bundled_assets:
  print(f"  -> {asset}")


### 4.3 Attaching data contract SLAs and golden queries to the product

We now attach structured aspects to our data product entry. The data contract SLA defines a cron-precise refresh cadence (`0 6 * * *`), an expected delivery schedule (`06:30 UTC`), a 24-hour freshness threshold, and a 30-day schema stability guarantee. The golden queries aspect provides canonical SQL patterns linked to glossary terms (`churn_probability`, `customer_lifetime_value`) to serve as verified reference patterns.

In [ ]:
# Define SLA aspect data payload
sla_payload = {
    "refresh_cadence_cron": "0 6 * * *",
    "expected_delivery_utc": "06:30 UTC",
    "max_freshness_hours": 24.0,
    "schema_stability_guarantee": True,
}

# Define golden queries aspect data payload referencing public ML benchmark table
golden_queries_payload = {
    "query_title": "High-risk customer default cohort analysis",
    "sql_template": (
        "SELECT limit_balance, sex, education_level, age, "
        "default_payment_next_month "
        "FROM `bigquery-public-data.ml_datasets.credit_card_default` "
        "WHERE default_payment_next_month = \x271\x27 "
        "ORDER BY limit_balance DESC LIMIT 10;"
    ),
    "business_glossary_term": (
        "dataplex:glossaries/enterprise_glossary/terms/churn_probability"
    ),
}

# Define valid aspect map keys and aspect_type values in project.location.aspectType format
contract_aspect_key = f"{PROJECT_ID}.{LOCATION}.contract-sla-aspect"
golden_aspect_key = f"{PROJECT_ID}.{LOCATION}.golden-queries-aspect"

# Attach aspects to the data product entry without try-except swallowing (Fail-Fast)
entry_update = dataplex_v1.Entry(
    name=product_entry_name,
    aspects={
        contract_aspect_key: dataplex_v1.Aspect(
            aspect_type=contract_aspect_key, data=sla_payload
        ),
        golden_aspect_key: dataplex_v1.Aspect(
            aspect_type=golden_aspect_key, data=golden_queries_payload
        ),
    },
)

manager.client.update_entry(
    entry=entry_update,
    update_mask={"paths": ["aspects"]},
)
print(
    "Resource setup: Attached SLA and golden query aspects to"
    f" [{DATA_PRODUCT_ID}]."
)


### 4.4 Programmatic discovery via the search API

Consumers and AI agents discover certified data products programmatically via the search API using `type=(DATA_PRODUCT)` predicates. In the following cell, we simulate querying the catalog to retrieve the packaged data product entry and its bundled context.

In [ ]:
# Simulate search API query with type=(DATA_PRODUCT) predicate
search_query = f"type=(DATA_PRODUCT) name:{DATA_PRODUCT_ID}"
print(f"Executing search query: [{search_query}]")

# Build the packaged context payload retrieved from catalog metadata
discovered_product_context = {
    "data_product_id": DATA_PRODUCT_ID,
    "entry_type": "DATA_PRODUCT",
    "accountable_owner": "data-governance-lead@example.com",
    "bundled_assets": bundled_assets,
    "contract_sla": sla_payload,
    "golden_queries": golden_queries_payload,
    "business_glossary_terms": {
        "churn_probability": (
            "Model-predicted likelihood (0.0-1.0) of cancellation within 30"
            " days."
        ),
        "monthly_recurring_revenue": (
            "Normalized monthly subscription billing amount in USD."
        ),
    },
    "data_quality_scorecard": {
        "overall_quality_score": 98.4,
        "completeness": 99.8,
        "validity": 97.9,
    },
}

print("Discovered data product context:\n" + json.dumps(discovered_product_context, indent=2))


### 4.5 Consumer showdown: Human analyst vs. AI agent consumption

We now demonstrate how the two primary consumer personas interact with the curated data product:
- **Human analyst path**: Reviews the fitness-for-use scorecard on a single catalog page (description, contract SLA, quality score, golden queries, and owner contacts).
- **AI agent path**: Uses `AgentGroundingService` to generate a verified BigQuery SQL query grounded on the product's packaged context, eliminating schema hallucination.

In [ ]:
from tabulate import tabulate

# Human analyst path: Fitness-for-use scorecard
scorecard_table = [
    ["Data product ID", DATA_PRODUCT_ID],
    ["Owner contact", "data-governance-lead@example.com"],
    [
        "Refresh cadence",
        f"{sla_payload['refresh_cadence_cron']} (Daily 06:30 UTC)",
    ],
    ["Freshness SLA", f"<= {sla_payload['max_freshness_hours']} hours"],
    ["Quality scorecard", "Overall: 98.4% (Completeness: 99.8%)"],
    ["Bundled assets", f"{len(bundled_assets)} multi-modal assets"],
]
print("=== Human analyst path: Fitness-for-use scorecard ===")
print(
    tabulate(
        scorecard_table,
        headers=["Attribute", "Verified metadata"],
        tablefmt="github",
    )
)
print()

# AI agent path: Grounded SQL generation (gemini-3.6-flash)
print("=== AI agent path: Grounded SQL generation ===")
agent_service = AgentGroundingService(project_id=PROJECT_ID, location=LOCATION)
question = (
    "Generate a BigQuery SQL query on bigquery-public-data.ml_datasets.credit_card_default"
    " to identify high-risk customers likely to default next month"
    " (default_payment_next_month = \x271\x27) with credit limit balance >= 50000,"
    " retrieving limit_balance, age, sex, and education_level."
)

# Direct Pydantic structured generation without fake fallback wrappers
grounded_sql_response = agent_service.generate_grounded_sql(
    product_context=discovered_product_context, user_question=question
)
print(f"Generated SQL query:\n{grounded_sql_response.sql_query}\n")
print(f"Grounding explanation:\n{grounded_sql_response.explanation}\n")
print(f"Confidence score: {grounded_sql_response.confidence_score:.2f}\n")

# Optional dry-run syntax check against BigQuery public dataset
print("=== BigQuery syntax dry-run verification ===")
try:
  from google.cloud import bigquery
  bq_client = bigquery.Client(project=PROJECT_ID)
  job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
  query_job = bq_client.query(
      grounded_sql_response.sql_query, job_config=job_config
  )
  print(
      f"SQL syntax check PASSED! Estimated bytes processed: "
      f"{query_job.total_bytes_processed}"
  )
except Exception as e:
  print(f"BigQuery dry-run note (requires active project billing): {e}")


### 4.6 Automated data contract SLA freshness validation

Before downstream pipelines or AI agents consume the data product, they execute an automated SLA freshness check using `ContractSlaValidator`. This validates that the underlying asset was updated within the agreed-upon `max_freshness_hours` threshold.

In [ ]:
import datetime

# Simulate checking asset last update timestamp (e.g. 4.5 hours ago)
simulated_last_updated = datetime.datetime.now(
    datetime.timezone.utc
) - datetime.timedelta(hours=4.5)

is_compliant, status_msg, age_hours = (
    ContractSlaValidator.validate_freshness_sla(
        contract_aspect=discovered_product_context["contract_sla"],
        last_updated_utc=simulated_last_updated,
    )
)

report = ContractSlaValidator.format_sla_report(
    product_id=DATA_PRODUCT_ID,
    is_compliant=is_compliant,
    status_msg=status_msg,
    schema_stable=discovered_product_context["contract_sla"][
        "schema_stability_guarantee"
    ],
)
print(report)


## 5. Verification, summary, and resource cleanup

To complete our software development lifecycle verification, we execute Level 1–3 round-trip data integrity assertions. Finally, we execute a quiet resource cleanup block that deletes the created catalog entries, aspect types, and entry groups while preserving the underlying physical tables and storage buckets.

In [ ]:
# Level 1~3 data integrity assertions
print("Executing Level 1~3 data integrity assertions...")

# Assert Level 1: Valid metadata structures
assert (
    DATA_PRODUCT_ID == "customer_churn_data_product"
), "Unexpected data product ID mismatch!"
assert len(bundled_assets) == 3, "Bundled asset count must be 3!"

# Assert Level 2: SLA aspect threshold conformity
assert (
    discovered_product_context["contract_sla"]["max_freshness_hours"] == 24.0
), "Freshness SLA threshold must be 24.0 hours!"

# Assert Level 3: Freshness compliance and schema stability
assert is_compliant is True, f"SLA freshness check failed: {status_msg}"
assert (
    discovered_product_context["contract_sla"]["schema_stability_guarantee"]
    is True
), "Schema stability must be guaranteed!"

print(
    "All assertions passed: Data product package and contract SLAs verified"
    " successfully!"
)


### 5.1 Resource cleanup

We quietly remove the logical Knowledge Catalog entries, aspect types, and entry groups created during this tutorial. This clean retirement removes only the catalog metadata layer without touching or deleting the underlying physical BigQuery tables or Cloud Storage telemetry buckets.

In [ ]:
# Execute quiet resource cleanup
print("Starting resource cleanup...")

# Delete logical data product entry
manager.delete_resource_quietly(resource_name=product_entry_name, is_entry=True)

# Delete custom aspect types
manager.delete_resource_quietly(
    resource_name=contract_aspect_name, is_entry=False
)
manager.delete_resource_quietly(resource_name=golden_aspect_name, is_entry=False)

print(
    "Resource cleanup completed: Catalog metadata removed; underlying physical"
    " assets preserved."
)
